# ConstructionSite XAI Pilot Walkthrough

This notebook is a guided walkthrough of the Florence-2-based explainable AI pipeline used in this project. It mirrors the repository workflow from dataset loading to model inference, visual inspection, and evaluation.

The goal is to make each stage transparent: what data is loaded, what annotations exist, what the model predicts, how we evaluate it, and how we visualize the outputs.

## 1. Setup and imports

The notebook assumes the project is installed in editable mode or that the repository root is on the Python path.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from PIL import Image
from IPython.display import display, Markdown

# Make the pilot package importable from the notebook directory.
current = Path.cwd()
pilot_root = None
for candidate in [current, current.parent, current.parent.parent]:
    if (candidate / 'src').exists() and (candidate / 'data').exists():
        pilot_root = candidate
        break
if pilot_root is None:
    raise RuntimeError('Could not locate the pilot project root. Please open this notebook from the pilot/ directory or adjust the path manually.')

sys.path.insert(0, str(pilot_root / 'src'))

from xai_pilot import config
from xai_pilot.config import DATA_DIR, RESULTS_DIR, FIGURES_DIR
from xai_pilot.data import classify_image, load_construction_site, select_balanced_sample
from xai_pilot.inference import answer_rule
from xai_pilot.model import load_florence2, run_task
from xai_pilot.viz import overlay_boxes, save_figure

print(f'Pilot root: {pilot_root}')
print(f'Config decoding: {config.DECODING}')
print(f'Data dir: {DATA_DIR}')

## 2. Load the dataset and inspect its schema

The project uses the ConstructionSite dataset from Hugging Face. When a local parquet snapshot exists, the loader prefers that copy.

In [ ]:
# Load a small slice of the test split to inspect the schema.
ds = load_construction_site(split='test', streaming=True)
rows = []
for row in ds:
    rows.append(row)
    if len(rows) >= 3:
        break

print('Loaded rows:', len(rows))
print('First row keys:', sorted(rows[0].keys()))
print('Image type:', type(rows[0]['image']))
print('Image id:', rows[0]['image_id'])
print('Image caption:', rows[0]['image_caption'][:180])
print('Annotation fields present:', [k for k in rows[0].keys() if 'violation' in k or k in {'excavator', 'rebar', 'worker_with_white_hard_hat'}])

# Show a compact dataframe of the first few rows.
meta_df = pd.DataFrame([{
    'image_id': r['image_id'],
    'image_caption': r['image_caption'][:80],
    'rule_1_violation': r.get('rule_1_violation'),
    'rule_2_violation': r.get('rule_2_violation'),
    'rule_3_violation': r.get('rule_3_violation'),
    'rule_4_violation': r.get('rule_4_violation'),
    'excavator': r.get('excavator'),
    'rebar': r.get('rebar'),
    'worker_with_white_hard_hat': r.get('worker_with_white_hard_hat')
} for r in rows])
display(meta_df)

## 3. Load pilot sample selection and prompt metadata

The pilot uses a balanced sample manifest and a set of per-rule grounding prompts. These are the inputs that define the experimental task.

In [ ]:
samples_df = pd.read_csv(DATA_DIR / 'pilot_samples.csv', dtype=str)
prompts = json.loads((DATA_DIR / 'safety_prompts.json').read_text(encoding='utf-8'))

display(Markdown('### Pilot sample manifest'))
display(samples_df.head(10))

display(Markdown('### Rule prompts'))
for rule_id, phrases in prompts.items():
    print(f'{rule_id}: {phrases}')

## 4. Basic dataset statistics and class distribution

This helps us see how balanced the sampled pilot set is and how the labels are distributed.

In [ ]:
# Count primary classes in the selected pilot sample manifest.
class_counts = samples_df['primary_class'].value_counts().reindex(['compliant', 'ppe_violation', 'fall_hazard', 'struck_by_risk'], fill_value=0)
class_counts.plot(kind='bar', figsize=(6, 4), color=['#4C78A8', '#F58518', '#54A24B', '#E45756'])
plt.title('Pilot sample class distribution')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

display(class_counts)

## 5. Visualize one image and its provided annotations

The dataset already contains annotation boxes. Here, we overlay them on the image for inspection.

In [ ]:
# Pull one sample from the test split by image_id.
sample_id = samples_df.iloc[0]['image_id']
sample_row = None
for row in load_construction_site(split='test', streaming=True):
    if row['image_id'] == sample_id:
        sample_row = row
        break
if sample_row is None:
    raise RuntimeError('Could not find requested sample in the test split')

image = sample_row['image']

# Collect non-empty annotation boxes from the dataset fields.
boxes_by_label = []
for key in ['excavator', 'rebar', 'worker_with_white_hard_hat']:
    value = sample_row.get(key, [])
    if value:
        boxes_by_label.extend([(box, key) for box in value])

if boxes_by_label:
    overlay = image.copy()
    for box, label in boxes_by_label:
        overlay = overlay_boxes(overlay, [box], labels=[label], color='red')
    display(overlay)
else:
    display(image)

print('Sample id:', sample_id)
print('Primary class:', samples_df.loc[samples_df['image_id'] == sample_id, 'primary_class'].iloc[0])
print('Assigned rule:', samples_df.loc[samples_df['image_id'] == sample_id, 'assigned_rule_id'].iloc[0])
print('Dataset annotations present:')
for key in ['excavator', 'rebar', 'worker_with_white_hard_hat']:
    print('-', key, '->', bool(sample_row.get(key)))

## 6. Load Florence-2 and run inference for one sample

This is the core model step: Florence-2 generates grounding boxes for worker-related and object-related phrases, and our code turns those into a rule verdict.

In [ ]:
# Load the model once. This can take time the first time because the weights are downloaded.
model, processor = load_florence2()

rule_id = samples_df.loc[samples_df['image_id'] == sample_id, 'assigned_rule_id'].iloc[0]
result = answer_rule(model, processor, image, rule_id)

print('Predicted answer:', result.answer)
print('Confidence:', result.confidence)
print('Inference ms:', result.inference_ms)
print('Worker boxes:', len(result.worker_boxes))
print('Object boxes:', len(result.object_boxes))

overlay = image.copy()
overlay = overlay_boxes(overlay, result.worker_boxes, labels=['worker'] * len(result.worker_boxes), color='blue')
overlay = overlay_boxes(overlay, result.object_boxes, labels=[rule_id] * len(result.object_boxes), color='red')
display(overlay)

## 7. Run the pipeline on a small batch of samples

The next cell mirrors the baseline inference script but on a handful of images so the notebook remains readable and fast.

In [ ]:
# Run a few samples and collect predictions into a dataframe.
sample_ids = samples_df['image_id'].head(6).tolist()
rows_out = []

for image_id in sample_ids:
    rule_id = samples_df.loc[samples_df['image_id'] == image_id, 'assigned_rule_id'].iloc[0]
    true_label = samples_df.loc[samples_df['image_id'] == image_id, 'primary_class'].iloc[0]
   
    # Re-load the image from the dataset stream for each id.
    ds_iter = load_construction_site(split='test', streaming=True)
    target_row = None
    for row in ds_iter:
        if row['image_id'] == image_id:
            target_row = row
            break
    if target_row is None:
        continue
    image = target_row['image']
    result = answer_rule(model, processor, image, rule_id)
    rows_out.append({
        'image_id': image_id,
        'true_label': true_label,
        'predicted_label': result.answer,
        'worker_boxes': len(result.worker_boxes),
        'object_boxes': len(result.object_boxes),
        'confidence': result.confidence,
        'inference_ms': result.inference_ms,
    })

pred_df = pd.DataFrame(rows_out)
display(pred_df)

## 8. Evaluate predictions against the dataset labels

This section turns the model outputs into simple accuracy and confusion-style summaries.

In [ ]:
# Convert labels to a consistent set of classes.
label_order = ['compliant', 'ppe_violation', 'fall_hazard', 'struck_by_risk']

def simple_confusion(true_labels, pred_labels, classes):
    rows = []
    for true_label in classes:
        row = []
        for pred_label in classes:
            row.append(sum(1 for t, p in zip(true_labels, pred_labels) if t == true_label and p == pred_label))
        rows.append(row)
    return pd.DataFrame(rows, index=classes, columns=classes)

true_labels = pred_df['true_label'].tolist()
pred_labels = pred_df['predicted_label'].tolist()

accuracy = sum(1 for t, p in zip(true_labels, pred_labels) if t == p) / len(pred_df)
print('Batch accuracy:', round(accuracy, 3))
display(simple_confusion(true_labels, pred_labels, label_order))

## 9. Optional: save a small set of overlay images

This produces visual artifacts that are useful for manual inspection, much like the project’s figure-generation step.

In [ ]:
overlay_dir = FIGURES_DIR / 'notebook_walkthrough'
overlay_dir.mkdir(parents=True, exist_ok=True)

for image_id in sample_ids[:3]:
    rule_id = samples_df.loc[samples_df['image_id'] == image_id, 'assigned_rule_id'].iloc[0]
    ds_iter = load_construction_site(split='test', streaming=True)
    target_row = None
    for row in ds_iter:
        if row['image_id'] == image_id:
            target_row = row
            break
    if target_row is None:
        continue
    image = target_row['image']
    result = answer_rule(model, processor, image, rule_id)
    overlay = image.copy()
    overlay = overlay_boxes(overlay, result.worker_boxes, labels=['worker'] * len(result.worker_boxes), color='blue')
    overlay = overlay_boxes(overlay, result.object_boxes, labels=['object'] * len(result.object_boxes), color='red')
    out_path = overlay_dir / f'{image_id}_{rule_id}.png'
    save_figure(overlay, out_path)
    print('Saved', out_path)

## 10. Interpretation and next steps

At this point the notebook has shown the main components of the pipeline:

1. Dataset loading and schema inspection
2. Sample selection and prompt setup
3. Visualization of dataset annotations
4. Florence-2-based grounding inference
5. Prediction evaluation and visual overlays

For a larger run, replace the small batch logic with a full pass over the selected pilot samples and save results to CSV, matching the repository’s numbered scripts.